
Imports all necessary Python libraries for data manipulation (`polars`, `pandas`, `numpy`), model building (`lightgbm`, `catboost`), and statistical calculations (`scipy`). Disables warnings for a clean output.

In [ ]:
import os
import numpy as np
import pandas as pd
import polars as pl
import joblib
import warnings
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from scipy.stats import spearmanr

warnings.filterwarnings('ignore')

Sets a `TRAIN` boolean flag and dynamically detects whether the code is running inside a Kaggle environment (standard kernel or scoring rerun) or locally, assigning directory paths accordingly.

In [ ]:
TRAIN = False

# Force inference-only inside the scoring container
IS_COMP_RUN = os.getenv('KAGGLE_IS_COMPETITION_RERUN') is not None
if IS_COMP_RUN:
    TRAIN = False

# Detect Kaggle environment
IS_KAGGLE = os.getenv('KAGGLE_KERNEL_RUN_TYPE') is not None

if IS_KAGGLE:
    import kaggle_evaluation.mitsui_inference_server
    DATA_PATH = '/kaggle/input/mitsui-commodity-prediction-challenge/'
    MODEL_INPUT_DIR = '/kaggle/input/v12-dataset/'
    MODEL_OUTPUT_DIR = '/kaggle/working/model-v12'
else:
    DATA_PATH = './kaggle/input/mitsui-commodity-prediction-challenge/'
    MODEL_INPUT_DIR = './model-v12/'
    MODEL_OUTPUT_DIR = './model-v12'

os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)

Defines global variables (columns to exclude, the list of all 424 targets, CV splits) and locks in the hyperparameters for the CatBoost and LightGBM models.

In [ ]:
EXCLUDE_COLS = {'date_id', 'row_id', 'is_scored'}
TARGETS = [f"target_{i}" for i in range(424)]
SOLUTION_NULL_FILLER = -999999
CV_SPLITS = 4

# Consistent hyperparameters for all targets
CATBOOST_PARAMS = {
    'iterations': 100,
    'depth': 5,
    'learning_rate': 0.05,
    'l2_leaf_reg': 5.0,
    'verbose': 0,
    'random_state': 42
}

LIGHTGBM_PARAMS = {
    'n_estimators': 100,
    'max_depth': 5,
    'learning_rate': 0.05,
    'reg_alpha': 0.5,
    'reg_lambda': 0.5,
    'random_state': 42,
    'verbose': -1
}

Loads the CSV files using Polars, sorts the time-series data chronologically by `date_id`, identifies common features between train and test sets, and creates a mapping dictionary for target lags.

In [ ]:
print("Loading data...")
train_df = pl.read_csv(os.path.join(DATA_PATH, 'train.csv'))
test_df = pl.read_csv(os.path.join(DATA_PATH, 'test.csv'))
train_labels = pl.read_csv(os.path.join(DATA_PATH, 'train_labels.csv'))
target_pairs = pl.read_csv(os.path.join(DATA_PATH, 'target_pairs.csv'))

print(f"Train: {train_df.shape}, Test: {test_df.shape}, Labels: {train_labels.shape}")

# Sort by date_id
if 'date_id' in train_df.columns:
    train_df = train_df.sort('date_id')
if 'date_id' in train_labels.columns:
    train_labels = train_labels.sort('date_id')

# Get common columns
COMMON_COLUMNS = [c for c in train_df.columns if c in set(test_df.columns)]
FEATURE_BASE = [c for c in COMMON_COLUMNS if c not in EXCLUDE_COLS]

# Create target-to-lag mapping
target_pairs_pd = target_pairs.to_pandas()
TARGET_LAG_MAP = dict(zip(target_pairs_pd['target'], target_pairs_pd['lag']))

### Leak-Free Feature Engineering



This cell applies financial transformations to the raw data to capture market momentum, volatility, and cross-asset relationships.
It executes a **shift-by-1-period** on all base columns *before* any rolling mathematics or aggregations occur. This shifting acts as a firewall against look-ahead bias (data leakage), ensuring that the features generated for a specific day only ever use information available prior to that day. 

Once the data is shifted, it computes several advanced indicators:
* **Cross-sectional aggregations & Rolling Statistics:** Captures the moving averages and standard deviations across asset classes to establish baseline trends.
* **Momentum Indicators (RSI & Rate of Change):** Measures the speed and magnitude of recent price changes to evaluate overbought or oversold conditions.
* **Cross-asset correlations & Spreads:** Calculates rolling correlations and z-scores for pairs like Gold/Platinum and Gold/Silver to identify divergence trading signals.
* **Parkinson Volatility & ATR (Average True Range):** Uses high/low price extremes to estimate market volatility more accurately than simple closing price variance.
* **Currency Carry Trade Indicators:** Evaluates average yields and stress across JPY currency pairs to capture macroeconomic risk sentiment.

The engineering of these specific features (without future leakage) makes the model more robust and more prone to identify patterns rather than memorizing the training set.

In [ ]:
def create_features(df: pl.DataFrame) -> pl.DataFrame:
    df_pd = df.to_pandas()
    
    print(f"Creating leak-free features from {df_pd.shape}...")
    
    # Shift all base columns at once (prevents confusion)
    base_cols_to_shift = [c for c in df_pd.columns if c not in ['date_id']]
    for col in base_cols_to_shift:
        df_pd[f'{col}_shifted'] = df_pd[col].shift(1)
    
    # Cross-sectional aggregations (no time leakage)
    asset_groups = {
        'LME': [f'{c}_shifted' for c in df_pd.columns if c.startswith('LME_') and c in base_cols_to_shift],
        'JPX': [f'{c}_shifted' for c in df_pd.columns if c.startswith('JPX_') and 'Close' in c and c in base_cols_to_shift],
        'US_Stock': [f'{c}_shifted' for c in df_pd.columns if c.startswith('US_Stock_') and 'adj_close' in c and c in base_cols_to_shift],
        'FX': [f'{c}_shifted' for c in df_pd.columns if c.startswith('FX_') and c in base_cols_to_shift]
    }
    
    for group_name, cols in asset_groups.items():
        if len(cols) > 0:
            df_pd[f'{group_name}_mean'] = df_pd[cols].mean(axis=1)
            df_pd[f'{group_name}_std'] = df_pd[cols].std(axis=1)
            df_pd[f'{group_name}_max'] = df_pd[cols].max(axis=1)
            df_pd[f'{group_name}_min'] = df_pd[cols].min(axis=1)
    
    # Rolling features on shifted data
    important_cols = [
        'JPX_Gold_Mini_Futures_Close', 'JPX_Platinum_Mini_Futures_Close',
        'US_Stock_GLD_adj_close', 'US_Stock_SLV_adj_close',
        'US_Stock_XLE_adj_close', 'US_Stock_XOM_adj_close',
        'FX_EURUSD', 'FX_USDJPY', 'FX_GBPUSD', 'FX_AUDUSD',
        'LME_CA_Close', 'LME_AH_Close', 'LME_ZS_Close',
        'US_Stock_GDX_adj_close',
    ]
    
    windows = [5, 10, 20, 30, 60]
    
    for col in important_cols:
        shifted_col = f'{col}_shifted'
        if shifted_col not in df_pd.columns:
            continue
        
        # Lagged values
        for lag in [3, 5, 8, 13, 21]:
            df_pd[f'{col}_lag_{lag}'] = df_pd[shifted_col].shift(lag - 1)
        
        # Rolling statistics
        for window in windows:
            df_pd[f'{col}_ma_{window}'] = df_pd[shifted_col].rolling(window, min_periods=max(1, window//2)).mean()
            df_pd[f'{col}_std_{window}'] = df_pd[shifted_col].rolling(window, min_periods=max(1, window//2)).std()
        
        # Rate of change
        df_pd[f'{col}_roc_10'] = df_pd[shifted_col].pct_change(10)
        df_pd[f'{col}_roc_20'] = df_pd[shifted_col].pct_change(20)
        
        # RSI
        delta = df_pd[shifted_col].diff()
        gain = (delta.where(delta > 0, 0)).rolling(14, min_periods=7).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(14, min_periods=7).mean()
        rs = gain / (loss + 1e-8)
        df_pd[f'{col}_rsi_14'] = 100 - (100 / (1 + rs))
    
    # Cross-asset correlations
    correlation_pairs = [
        ('JPX_Gold_Mini_Futures_Close', 'JPX_Platinum_Mini_Futures_Close'),
        ('US_Stock_GLD_adj_close', 'US_Stock_SLV_adj_close'),
        ('US_Stock_XLE_adj_close', 'US_Stock_XOM_adj_close'),
        ('US_Stock_GLD_adj_close', 'FX_EURUSD'),
        ('JPX_Gold_Mini_Futures_Close', 'FX_USDJPY'),
    ]
    
    for col1, col2 in correlation_pairs:
        col1_shifted = f'{col1}_shifted'
        col2_shifted = f'{col2}_shifted'
        
        if col1_shifted in df_pd.columns and col2_shifted in df_pd.columns:
            for window in [30, 60]:
                corr = df_pd[col1_shifted].rolling(window, min_periods=window//3).corr(df_pd[col2_shifted])
                df_pd[f'corr_{col1}_{col2}_{window}d'] = corr
        
    # Gold/Platinum ratio
    if 'JPX_Gold_Mini_Futures_Close_shifted' in df_pd.columns and 'JPX_Platinum_Mini_Futures_Close_shifted' in df_pd.columns:
        ratio = df_pd['JPX_Gold_Mini_Futures_Close_shifted'] / (df_pd['JPX_Platinum_Mini_Futures_Close_shifted'] + 1e-8)
        df_pd['Gold_Platinum_ratio'] = ratio
        
        ratio_mean = ratio.rolling(60, min_periods=20).mean()
        ratio_std = ratio.rolling(60, min_periods=20).std()
        df_pd['Gold_Platinum_ratio_zscore'] = (ratio - ratio_mean) / (ratio_std + 1e-8)
    
    # Gold/Silver spread
    if 'US_Stock_GLD_adj_close_shifted' in df_pd.columns and 'US_Stock_SLV_adj_close_shifted' in df_pd.columns:
        spread = df_pd['US_Stock_GLD_adj_close_shifted'] - df_pd['US_Stock_SLV_adj_close_shifted']
        df_pd['GLD_SLV_spread'] = spread
        
        spread_mean = spread.rolling(60, min_periods=20).mean()
        spread_std = spread.rolling(60, min_periods=20).std()
        df_pd['GLD_SLV_spread_zscore'] = (spread - spread_mean) / (spread_std + 1e-8)
    
    # Metal ratios
    metal_pairs = [
        ('LME_CA_Close', 'LME_AH_Close'),
        ('LME_ZS_Close', 'LME_PB_Close'),
    ]
    
    for metal1, metal2 in metal_pairs:
        m1_shifted = f'{metal1}_shifted'
        m2_shifted = f'{metal2}_shifted'
        
        if m1_shifted in df_pd.columns and m2_shifted in df_pd.columns:
            ratio = df_pd[m1_shifted] / (df_pd[m2_shifted] + 1e-8)
            df_pd[f'{metal1}_{metal2}_ratio'] = ratio
            
            spread = df_pd[m1_shifted] - df_pd[m2_shifted]
            df_pd[f'{metal1}_{metal2}_spread'] = spread
    
    # Volatility features
    for asset in ['JPX_Gold_Mini_Futures', 'JPX_Platinum_Mini_Futures', 'US_Stock_GLD', 'US_Stock_XLE']:
        if asset.startswith('JPX'):
            high_col = f'{asset}_High_shifted'
            low_col = f'{asset}_Low_shifted'
            close_col = f'{asset}_Close_shifted'
        else:
            high_col = f'{asset}_adj_high_shifted'
            low_col = f'{asset}_adj_low_shifted'
            close_col = f'{asset}_adj_close_shifted'
        
        if all(col in df_pd.columns for col in [high_col, low_col, close_col]):
            # Parkinson volatility
            log_hl = np.log((df_pd[high_col] + 1e-8) / (df_pd[low_col] + 1e-8))
            df_pd[f'{asset}_parkinson_vol_20'] = np.sqrt(
                1/(4*np.log(2)) * (log_hl**2).rolling(20, min_periods=10).mean()
            )
            
            # ATR
            atr = (df_pd[high_col] - df_pd[low_col]).rolling(14, min_periods=7).mean()
            df_pd[f'{asset}_atr_14'] = atr
            df_pd[f'{asset}_atr_norm'] = atr / (df_pd[close_col] + 1e-8)
    
    # Volume features
    volume_cols = [c for c in df_pd.columns if ('volume' in c.lower() or 'Volume' in c) and '_shifted' in c]
    
    for vol_col in volume_cols[:10]:
        # Volume momentum
        df_pd[f'{vol_col}_roc_10'] = df_pd[vol_col].pct_change(10)
        
        # Volume MA ratio
        vol_ma_10 = df_pd[vol_col].rolling(10, min_periods=5).mean()
        df_pd[f'{vol_col}_vs_ma10'] = df_pd[vol_col] / (vol_ma_10 + 1e-8)
    
    # Currency carry trade indicators
    jpy_pairs = [f'{col}_shifted' for col in df_pd.columns if col.startswith('FX_') and col.endswith('JPY')]
    
    if len(jpy_pairs) >= 3:
        jpy_avg = df_pd[jpy_pairs].mean(axis=1)
        df_pd['JPY_pairs_avg'] = jpy_avg
        df_pd['JPY_carry_momentum'] = jpy_avg.pct_change(10)
        df_pd['JPY_carry_stress'] = df_pd[jpy_pairs].std(axis=1)
    
    # Time-based features
    if 'date_id' in df_pd.columns:
        df_pd['date_id_mod_7'] = df_pd['date_id'] % 7
        df_pd['date_id_mod_30'] = df_pd['date_id'] % 30
        
        for period in [7, 30, 90]:
            df_pd[f'sin_date_{period}'] = np.sin(2 * np.pi * df_pd['date_id'] / period)
            df_pd[f'cos_date_{period}'] = np.cos(2 * np.pi * df_pd['date_id'] / period)
    
    # Remove shifted columns (keep only derived features)
    cols_to_drop = [c for c in df_pd.columns if c.endswith('_shifted')]
    df_pd = df_pd.drop(columns=cols_to_drop)
    
    print(f"Features created. Final shape: {df_pd.shape}")
    
    return pl.from_pandas(df_pd)

### Data Cleansing Preprocessor

This cell serves as a data sanitizer to ensure the feature matrix is perfectly formatted before it touches the machine learning algorithms. Both LightGBM and CatBoost handle `NaN` (missing) values natively, but will fail if fed infinite floating-point numbers or unmapped object strings. 

To prevent this, the preprocessor performs those three cleaning steps:
1.  **Type Correction:** It hardcodes a fix for the `US_Stock_GOLD` columns, which are known to occasionally parse as string objects in this specific dataset, forcefully coercing them into numeric formats.
2.  **Categorical Encoding:** It scans for any remaining categorical string columns and factorizes them into integer codes.
3.  **Infinity Sanitization:** It replaces any mathematical infinities (which often result from dividing by near-zero values during ratio calculations in the previous step) with standard `NaN` values, allowing the gradient boosters to process them safely.

In [ ]:
def preprocess_for_model(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    # Handle GOLD stock columns (known issue)
    for col in ['US_Stock_GOLD_adj_open', 'US_Stock_GOLD_adj_high',
                'US_Stock_GOLD_adj_low', 'US_Stock_GOLD_adj_close',
                'US_Stock_GOLD_adj_volume']:
        if col in df.columns and df[col].dtype == 'object':
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Handle categorical columns
    for cat_col in df.select_dtypes(['category']).columns:
        df[cat_col] = df[cat_col].cat.codes
    
    # Replace inf with NaN
    df = df.replace([np.inf, -np.inf], np.nan)
    
    return df

### Advanced Feature Selection



Financial datasets are noisy. I observed that feeding hundreds of engineered features directly into a model leads to severe overfitting. This cell implements three-tier reduction strategy to distill the feature set down to the most stable and predictive signals:

1.  **Adversarial Validation (Stability Pruning):** It trains a dummy classifier to try and distinguish between the training data and the test data. If a feature makes it too easy to tell train from test, it means the feature's underlying distribution changes too much over time (it is unstable). The cell identifies and drops the top 15% most unstable features, protecting the model from temporal overfitting.
2.  **Correlation Pruning (Redundancy Removal):** It builds a correlation matrix and drops features that have a >0.95 Pearson correlation with each other. This reduces extreme collinearity, making the ensemble models faster to train and easier to interpret without losing predictive power.
3.  **Importance Ranking (Signal Pruning):** Finally, it fits a preliminary LightGBM model on a sample of the targets to calculate feature importances. It ranks the surviving features and selects only the top 200 (or `n_features`) so that the final production models are trained only on the highest-quality, highest-signal data.

In [ ]:
def select_stable_features(X_all, y_all, test_df_features, n_features=200):
    from sklearn.metrics import roc_auc_score
    from lightgbm import LGBMClassifier
    
    print("\n======== FEATURE SELECTION ========")
    
    all_features = [c for c in X_all.columns if c not in EXCLUDE_COLS]
    
    # Step 1: Adversarial validation - identify unstable features
    print("  Step 1: Identifying unstable features...")
    
    train_features = X_all[all_features].copy()
    test_features = test_df_features[all_features].copy()
    
    train_features['is_test'] = 0
    test_features['is_test'] = 1
    
    combined = pd.concat([train_features, test_features], ignore_index=True)
    
    X_adv = combined[all_features].copy()
    X_adv = preprocess_for_model(X_adv).fillna(0)
    y_adv = combined['is_test']
    
    lgb_adv = LGBMClassifier(n_estimators=100, max_depth=5, random_state=42, verbose=-1)
    lgb_adv.fit(X_adv, y_adv)
    
    auc = roc_auc_score(y_adv, lgb_adv.predict_proba(X_adv)[:, 1])
    print(f"  Adversarial AUC: {auc:.4f}")
    
    # Remove top 15% most discriminative features
    adv_importance = pd.DataFrame({
        'feature': all_features,
        'adv_importance': lgb_adv.feature_importances_
    }).sort_values('adv_importance', ascending=False)
    
    n_unstable = max(10, len(all_features) // 7)  # Remove ~15%
    unstable_features = set(adv_importance.head(n_unstable)['feature'].tolist())
    print(f"  Removed {len(unstable_features)} unstable features")
    
    stable_features = [f for f in all_features if f not in unstable_features]
    
    # Step 2: Remove highly correlated features
    print("  Step 2: Removing correlated features...")
    
    X_sample = X_all[stable_features].iloc[::10].copy()  # Sample for speed
    X_sample = preprocess_for_model(X_sample).fillna(0)
    
    corr_matrix = X_sample.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]
    
    print(f"  Removed {len(to_drop)} correlated features")
    stable_features = [f for f in stable_features if f not in to_drop]
    
    # Step 3: Select top features by importance
    print("  Step 3: Selecting by predictive importance...")
    
    importance_scores = {}
    sample_targets = [f'target_{i}' for i in range(0, 424, 30)]  # Sample targets
    
    for tgt in sample_targets:
        y = y_all[tgt]
        mask = ~y.isna()
        if mask.sum() < 100:
            continue
        
        X_train = X_all.loc[mask, stable_features].copy()
        y_train = y.loc[mask]
        
        X_train = preprocess_for_model(X_train).fillna(0)
        
        lgb_temp = LGBMRegressor(n_estimators=50, max_depth=5, random_state=42, verbose=-1)
        lgb_temp.fit(X_train, y_train)
        
        for feat, imp in zip(X_train.columns, lgb_temp.feature_importances_):
            importance_scores[feat] = importance_scores.get(feat, 0) + imp
    
    # Select top N features
    sorted_features = sorted(importance_scores.items(), key=lambda x: x[1], reverse=True)
    selected_features = [f[0] for f in sorted_features[:n_features]]
    
    print(f"\n  Final selection: {len(selected_features)} features")
    
    return selected_features, unstable_features

### Validation Splitter & Scoring Metrics

This cell defines a local evaluation framework for our financial forecasting. Standard k-fold cross-validation fails on time-series data because the targets forecast returns several days into the future, creating overlapping horizons (target leakage). 

To solve this, the generator implements **Purged Cross-Validation with Embargo**: it strictly drops (purges) training samples if their forecast windows overlap with the validation set's timeline, preventing the model from memorizing the future. 

Alongside this, it defines the `rankcorr_sharpe` function, which computes the Sharpe ratio of daily Spearman rank correlations. This custom metric aims to mirror Kaggle's backend evaluation, to get the most accurate and trustworthy local validation score to guide the feature engineering.

In [ ]:
def purged_cv_split(df_length, date_ids, target_lag_map, n_splits=4):
    """Time series CV with purging and embargo."""
    embargo_days = max(target_lag_map.values()) + 1
    fold_size = df_length // n_splits
    
    for fold in range(n_splits):
        test_start_idx = fold * fold_size
        test_end_idx = min((fold + 1) * fold_size, df_length)
        
        test_start_date = date_ids[test_start_idx]
        test_end_date = date_ids[test_end_idx - 1]
        
        train_mask = np.ones(df_length, dtype=bool)
        
        # Purging: remove samples whose labels overlap with test period
        for idx in range(df_length):
            current_date = date_ids[idx]
            for lag in target_lag_map.values():
                label_date = current_date + lag + 1
                if label_date >= test_start_date and label_date <= test_end_date + embargo_days:
                    train_mask[idx] = False
                    break
        
        train_idx = np.where(train_mask)[0]
        test_idx = np.arange(test_start_idx, test_end_idx)
        
        if len(train_idx) >= 100:
            yield train_idx, test_idx

# Metric

def rankcorr_sharpe(preds_df: pd.DataFrame, truths_df: pd.DataFrame, 
                    filler: float = SOLUTION_NULL_FILLER) -> float:
    """Competition metric: Sharpe ratio of daily Spearman correlations."""
    P = preds_df.copy().ffill().bfill()
    T = truths_df.copy().fillna(filler)
    
    daily_scores = []
    for p_row, t_row in zip(P.values, T.values):
        mask = (t_row != filler) & np.isfinite(p_row)
        if mask.sum() < 2:
            daily_scores.append(0.0)
            continue
        
        p_rank = pd.Series(p_row[mask]).rank(method='average').to_numpy()
        t_rank = pd.Series(t_row[mask]).rank(method='average').to_numpy()
        
        if np.std(p_rank) == 0 or np.std(t_rank) == 0:
            daily_scores.append(0.0)
        else:
            daily_scores.append(np.corrcoef(p_rank, t_rank)[0, 1])
    
    arr = np.asarray(daily_scores, dtype=float)
    std = arr.std(ddof=0)
    return float(arr.mean() / std) if std > 0 else 0.0

### Model State Management (Lazy Loading)

This block initializes global dictionaries to hold the trained model objects and selected feature lists, but it deliberately uses a `lazy_load_models` function to fetch the `.pkl` files from the disk only when the first prediction is actually requested. Loading over 800 discrete models (424 CatBoost + 424 LightGBM) into memory all at once during script initialization sometimes triggers an Out-Of-Memory (OOM) crash in the Kaggle environment.

In [ ]:
models_cb = {}
models_lgb = {}
MODEL_FEATURES = {}
SELECTED_FEATURES = None
MODELS_LOADED = False

def lazy_load_models():
    """Load models on first inference call."""
    global MODELS_LOADED, models_cb, models_lgb, MODEL_FEATURES, SELECTED_FEATURES
    
    if MODELS_LOADED:
        return
    
    print("Loading models...")
    
    # Load selected features list
    feat_list_path = os.path.join(MODEL_INPUT_DIR, 'selected_features.pkl')
    if os.path.exists(feat_list_path):
        SELECTED_FEATURES = joblib.load(feat_list_path)
        print(f"Loaded {len(SELECTED_FEATURES)} selected features")
    else:
        raise FileNotFoundError(f"selected_features.pkl not found in {MODEL_INPUT_DIR}")
    
    loaded_cb = 0
    loaded_lgb = 0
    
    for tgt in TARGETS:
        # Load CatBoost
        cb_path = os.path.join(MODEL_INPUT_DIR, f"{tgt}_cb.pkl")
        if os.path.exists(cb_path):
            models_cb[tgt] = joblib.load(cb_path)
            loaded_cb += 1
        else:
            models_cb[tgt] = None
        
        # Load LightGBM
        lgb_path = os.path.join(MODEL_INPUT_DIR, f"{tgt}_lgb.pkl")
        if os.path.exists(lgb_path):
            models_lgb[tgt] = joblib.load(lgb_path)
            loaded_lgb += 1
        else:
            models_lgb[tgt] = None
        
        # Load features
        feat_path = os.path.join(MODEL_INPUT_DIR, f"{tgt}_feat.pkl")
        if os.path.exists(feat_path):
            MODEL_FEATURES[tgt] = joblib.load(feat_path)
        else:
            MODEL_FEATURES[tgt] = None
    
    MODELS_LOADED = True
    print(f"Loaded {loaded_cb} CatBoost and {loaded_lgb} LightGBM models")

### Main Training Routine

When the `TRAIN` flag is set to True, this cell orchestrates the entire modeling lifecycle. It generates the full feature set, runs the adversarial feature selection, and loops through the 4-fold purged cross-validation to calculate an Out-Of-Fold (OOF) Sharpe ratio. Once validated, it retrains the final CatBoost and LightGBM models on 100% of the available data, ensembles their predictions with a 60/40 split, and pickles the final artifacts to the output directory. 

Blending those two distinct gradient boosting architectures (CatBoost's oblivious trees vs. LightGBM's leaf-wise growth) helped diversifie the algorithmic risk, smoothing out extreme predictions and lead to an overall better Sharpe ratio. Retraining on the full dataset at the end ensures the production models have absorbed the complete historical context available.

In [ ]:
if TRAIN:
    print("\n======== TRAINING MODE ========")
    
    # Create features
    X_all_df = create_features(train_df).to_pandas()
    y_all = train_labels.to_pandas()[TARGETS]
    
    # Feature selection
    test_df_features = create_features(test_df).to_pandas()
    SELECTED_FEATURES, unstable_features = select_stable_features(
        X_all_df, y_all, test_df_features, n_features=200
    )
    
    X_all = X_all_df[SELECTED_FEATURES]
    date_ids = train_df['date_id'].to_numpy()
    
    # Save feature list
    joblib.dump(SELECTED_FEATURES, os.path.join(MODEL_OUTPUT_DIR, 'selected_features.pkl'))
    
    # Purged CV validation
    print(f"\n======== PURGED CV (k={CV_SPLITS}) ========")
    
    oof = pd.DataFrame(index=y_all.index, columns=TARGETS, dtype=float)
    fold_scores = []
    
    for fold, (tr_idx, va_idx) in enumerate(purged_cv_split(
        len(X_all), date_ids, TARGET_LAG_MAP, n_splits=CV_SPLITS
    )):
        print(f"\n  Fold {fold+1}/{CV_SPLITS}")
        print(f"    Train: {len(tr_idx)}, Val: {len(va_idx)}")
        
        X_tr_fold = X_all.iloc[tr_idx]
        X_va_fold = X_all.iloc[va_idx]
        y_tr_fold = y_all.iloc[tr_idx]
        
        for idx, tgt in enumerate(TARGETS):
            if idx % 100 == 0:
                print(f"    Target {idx}/424")
            
            y_tr = y_tr_fold[tgt]
            mask = ~y_tr.isna()
            
            if mask.sum() < 10:
                continue
            
            X_tr = X_tr_fold.loc[mask].copy()
            y_tr_clean = y_tr.loc[mask]
            X_va = X_va_fold.copy()
            
            # Add target encoding
            X_tr['target_name'] = idx
            X_va['target_name'] = idx
            
            feat_order = SELECTED_FEATURES + ['target_name']
            X_tr = preprocess_for_model(X_tr).fillna(0).reindex(columns=feat_order, fill_value=0.0)
            X_va = preprocess_for_model(X_va).fillna(0).reindex(columns=feat_order, fill_value=0.0)
            
            # Train CatBoost
            m_cb = CatBoostRegressor(**CATBOOST_PARAMS)
            m_cb.fit(X_tr, y_tr_clean)
            pred_cb = m_cb.predict(X_va)
            
            # Train LightGBM
            m_lgb = LGBMRegressor(**LIGHTGBM_PARAMS)
            m_lgb.fit(X_tr, y_tr_clean)
            pred_lgb = m_lgb.predict(X_va)
            
            # Ensemble: 60% CatBoost + 40% LightGBM
            oof.loc[va_idx, tgt] = 0.6 * pred_cb + 0.4 * pred_lgb
        
        # Fold score
        y_scored = y_all.fillna(SOLUTION_NULL_FILLER)
        fold_score = rankcorr_sharpe(oof.loc[va_idx], y_scored.loc[va_idx], SOLUTION_NULL_FILLER)
        fold_scores.append(fold_score)
        print(f"    Fold {fold+1} Sharpe: {fold_score:.4f}")
    
    # Overall OOF score
    cv_score = rankcorr_sharpe(oof, y_scored, SOLUTION_NULL_FILLER)
    
    print("\n" + "="*60)
    print("CV RESULTS")
    print("="*60)
    print(f"OOF Sharpe: {cv_score:.4f}")
    print(f"Mean Fold Sharpe: {np.mean(fold_scores):.4f}")
    print(f"Std Fold Sharpe: {np.std(fold_scores):.4f}")
    print("="*60)
    
    # Train final models on full data
    print("\n======== TRAINING FINAL MODELS ========")
    
    for idx, tgt in enumerate(TARGETS):
        if idx % 50 == 0:
            print(f"  Training target {idx}/{len(TARGETS)}")
        
        y = y_all[tgt]
        mask = ~y.isna()
        
        if mask.sum() < 10:
            models_cb[tgt] = None
            models_lgb[tgt] = None
            MODEL_FEATURES[tgt] = SELECTED_FEATURES + ['target_name']
            continue
        
        X_tr = X_all.loc[mask].copy()
        y_tr = y.loc[mask]
        
        # Add target encoding
        X_tr['target_name'] = idx
        
        feat_order = SELECTED_FEATURES + ['target_name']
        X_tr = preprocess_for_model(X_tr).fillna(0).reindex(columns=feat_order, fill_value=0.0)
        
        # Train CatBoost
        m_cb = CatBoostRegressor(**CATBOOST_PARAMS)
        m_cb.fit(X_tr, y_tr)
        models_cb[tgt] = m_cb
        
        # Train LightGBM
        m_lgb = LGBMRegressor(**LIGHTGBM_PARAMS)
        m_lgb.fit(X_tr, y_tr)
        models_lgb[tgt] = m_lgb
        
        MODEL_FEATURES[tgt] = feat_order
        
        # Save models
        joblib.dump(m_cb, os.path.join(MODEL_OUTPUT_DIR, f"{tgt}_cb.pkl"))
        joblib.dump(m_lgb, os.path.join(MODEL_OUTPUT_DIR, f"{tgt}_lgb.pkl"))
        joblib.dump(feat_order, os.path.join(MODEL_OUTPUT_DIR, f"{tgt}_feat.pkl"))
    
    print(f"\nModels saved to: {MODEL_OUTPUT_DIR}")
    print("\n" + "="*60)
    print("TRAINING COMPLETE")
    print("="*60)
    print(f"Next steps:")
    print(f"1. Copy models from {MODEL_OUTPUT_DIR} to your Kaggle dataset")
    print(f"2. Update MODEL_INPUT_DIR to point to that dataset")
    print(f"3. Set TRAIN=False and submit")
    print("="*60)

else:
    print("\n======== INFERENCE MODE ========")
    print(f"Models will be loaded from: {MODEL_INPUT_DIR}")
    
    # Check if models exist
    if not os.path.exists(MODEL_INPUT_DIR):
        raise FileNotFoundError(f"MODEL_INPUT_DIR does not exist: {MODEL_INPUT_DIR}")
    
    feat_list_path = os.path.join(MODEL_INPUT_DIR, 'selected_features.pkl')
    if not os.path.exists(feat_list_path):
        raise FileNotFoundError(f"selected_features.pkl not found. Did you train the model first?")
    
    print(f"Found selected_features.pkl ✓")

### Inference Callback API

This block defines the exact `predict` callback required by the Kaggle Evaluation API. When the server yields a batch of new test data, this function engineers the required features, aligns the dataframe columns to match the trained model's expected input structure, and executes the 60/40 ensembled predictions. 

In [ ]:
def predict(
    test: pl.DataFrame,
    label_lags_1_batch: pl.DataFrame,
    label_lags_2_batch: pl.DataFrame,
    label_lags_3_batch: pl.DataFrame,
    label_lags_4_batch: pl.DataFrame,
) -> pd.DataFrame:
    """Generate predictions for test set."""
    lazy_load_models()
    
    # Create features
    df_feat = create_features(test).to_pandas()
    
    # Use SELECTED_FEATURES (loaded by lazy_load_models)
    if SELECTED_FEATURES is None:
        raise ValueError("SELECTED_FEATURES not loaded")
    
    # Get only the selected features that exist in df_feat
    available_selected = [f for f in SELECTED_FEATURES if f in df_feat.columns]
    
    if len(available_selected) < len(SELECTED_FEATURES) * 0.9:
        print(f"Warning: Only {len(available_selected)}/{len(SELECTED_FEATURES)} features available")
    
    X_base = df_feat[available_selected].copy()
    X_base = preprocess_for_model(X_base).fillna(0)
    
    n_rows = len(X_base)
    predictions = np.zeros((n_rows, len(TARGETS)), dtype=np.float64)
    
    for idx, tgt in enumerate(TARGETS):
        cb_model = models_cb.get(tgt)
        lgb_model = models_lgb.get(tgt)
        feat_order = MODEL_FEATURES.get(tgt)
        
        if cb_model is None and lgb_model is None:
            predictions[:, idx] = 0.0
            continue
        
        if feat_order is None:
            feat_order = available_selected + ['target_name']
        
        # Prepare features
        X_pred = X_base.copy()
        X_pred['target_name'] = idx
        
        # Ensure all required features exist
        for f in feat_order:
            if f not in X_pred.columns:
                X_pred[f] = 0.0
        
        X_pred = X_pred.reindex(columns=feat_order, fill_value=0.0)
        
        # Predict with both models
        pred_cb = cb_model.predict(X_pred) if cb_model is not None else np.zeros(n_rows)
        pred_lgb = lgb_model.predict(X_pred) if lgb_model is not None else np.zeros(n_rows)
        
        # Ensemble: 60% CatBoost + 40% LightGBM
        pred = 0.6 * pred_cb + 0.4 * pred_lgb
        
        # Ensure correct shape
        if len(pred) != n_rows:
            pred = np.resize(pred, n_rows)
        
        predictions[:, idx] = pred
    
    # Handle NaN/inf
    predictions = np.nan_to_num(predictions, nan=0.0, posinf=0.0, neginf=0.0)
    
    # Convert to DataFrame with correct column names
    result = pd.DataFrame(predictions, columns=TARGETS)
    
    # Final validation
    assert result.shape[0] == n_rows, f"Wrong number of rows: {result.shape[0]} != {n_rows}"
    assert result.shape[1] == 424, f"Wrong number of columns: {result.shape[1]} != 424"
    assert list(result.columns) == TARGETS, "Column names don't match"
    assert np.isfinite(result.to_numpy()).all(), "Non-finite values in predictions"
    assert result.isna().sum().sum() == 0, "NaN values in predictions"
    
    return result

### Kaggle API Bootstrapper

This final cell is the official execution trigger. It instantiates the `MitsuiInferenceServer` and hooks the custom `predict` logic into it. If I run the notebook locally (or in an interactive Kaggle session), it injects dummy data arrays to simulate the API, validating the pipeline's structural integrity and generating a local `submission_local.csv`. If it detects the hidden competition rerun environment, it hands control over to Kaggle's remote evaluation server. This dual-mode design allows makes it saffer to debug the logic locally while keeping the script completely competition-ready.

In [ ]:
if IS_KAGGLE:
    server = kaggle_evaluation.mitsui_inference_server.MitsuiInferenceServer(predict)
    if IS_COMP_RUN:
        server.serve()
    else:
        server.run_local_gateway((DATA_PATH,))
else:
    # Local testing
    print("\n======== GENERATING LOCAL SUBMISSION ========")
    
    mock = pl.DataFrame()
    submission = predict(test_df, mock, mock, mock, mock)
    
    print(f"Submission shape: {submission.shape}")
    print(f"NaN values: {submission.isna().sum().sum()}")
    print(f"Range: [{submission.min().min():.4f}, {submission.max().max():.4f}]")
    
    submission.to_csv("submission_local.csv", index=False)
    print("\nLocal submission saved -> submission_local.csv")